In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("Occurence-bronze").getOrCreate()

# I think first ill ingest all available then ill add batch process that will run once / twice a month
snapshot_date_str = "2021-04-13"

df = spark.read.format("parquet").load(
    f"s3a://gbif-open-data-af-south-1/occurrence/{snapshot_date_str}/occurrence.parquet/"
)

ph_filtered = df.filter(F.col("countrycode") == "PH")

ph_filtered = (
    ph_filtered.withColumn(
        "ingestion_timestamp", 
        F.current_timestamp()
    )
    .withColumn(
        "source_file_path", 
        F.col("_metadata.file_path")
    )
    .withColumn(
        "snapshot_date", 
        F.lit(snapshot_date_str).cast("date")  # Extracts the date from the folder name
    )
)

ph_filtered.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable("philippinebiodiveristyplatform.bronze.occurrence")